## Map dataset test

import libs

In [ ]:
import matplotlib.pyplot as plt

import cv2
import numpy as np
from pyproj import Transformer

from uavcalibration.datasets import *
from uavcalibration.map import *

Init map dataset

Please change the source of Tile Map to any api or folder available for you

In [ ]:
dataset = VisLocDataset(r"..\datasets\UAV_VisLoc_example")
geotiff_map = GeoTiffMap([i.image_path for i in dataset.satellite_infos.values()])
tile_map = TileMap(
    r"http://mt0.google.com/vt/lyrs=s&hl=en&x={x}&y={y}&z={z}",
    # r"..\datasets\UAV_VisLoc_example\03\tiles\{z}\{x}\{y}.png",
)

## Test coordinate conversion
Set map area

In [ ]:
bounds = (119.805927, 32.345490, 119.815927, 32.355490)
limits = (180, 85.051129, -180, -85.051129)
wgs = np.array([*bounds, *limits]).reshape(-1, 2).T
zoom = 18
wgs2merc = Transformer.from_crs("epsg:4326", "epsg:3857", always_xy=True)
merc_xy = np.array(wgs2merc.transform(*wgs))
merc_xy

Convert web mercator coordinates to tile and pixel coordinates

In [ ]:
global_xy = tile_map.merc2tile(*merc_xy, zoom=zoom).astype(np.uint64)
tile_xy, pixel_xy = np.divmod(global_xy, tile_map.tile_size)
(tile_xy, pixel_xy)

## Get Image

In [ ]:
async with geotiff_map:
    image, trans = await geotiff_map.get_async(bounds,resolution=1e-5)
plt.imshow(image)
trans

In [ ]:
async with tile_map:
    image, trans = await tile_map.get_async(bounds,resolution=1e-5)
plt.imshow(image)
trans